# ETL meetpunten en edges ring van Antwerpen

Het doel van deze notebook is het ophalen van 2 datesets (meetpunten miv en edges argis portaal antwerpen) via API's. Daarna deze te parsen in dataframes en opladen in de Postgres database.  

## 1. ETL data

In [338]:
import numpy as np
#import folium 
import requests 
import xml.etree.ElementTree as ET 
import pandas as pd
import geopandas as gpd
import math
import psycopg2
import os
from dotenv import load_dotenv

### 1.1 Meetpunten

Eerst zal ik de meetpunt data van het miv ophalen via een API met XML formaat. Deze zal ik dan parsen om daarna een dataframe te maken met de gewenste data. 

In [339]:
url = "https://miv.opendata.belfla.be/miv/configuratie/xml"
response = requests.get(url, verify=False)
print(response)

/usr/local/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'miv.opendata.belfla.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


<Response [200]>


In [340]:
root = ET.fromstring(response.text)

In [341]:
def xml_to_dict(r):
    """
    Functie om de XML string om de zetten naar een dictionairy
    """

    # Maak een dictionary voor het huidige element
    result = {}

    # Voeg de attributen van het element toe als ze bestaan, belangrijk voor de unieke ID van meetpunten. 
    if r.attrib:
        result.update(('@' + k, v) for k, v in r.attrib.items())

    # Als het element geen kinderen heeft, voeg de tekst toe
    if len(list(r)) == 0:
        result[r.tag] = r.text
    else:
        # Anders voeg de kinderen als een lijst van dictionaries toe
        result[r.tag] = [xml_to_dict(child) for child in r]

    return result
    
d = xml_to_dict(root)

Lege dataframe opstellen met gewenste info

In [342]:
df_meetpunten = pd.DataFrame(
        {'unieke_id': [],
         'beschrijvende_id': [],
         'volledige_naam': [],
         'Ident_8': [],
         'Rijstrook': [],
         'lengtegraad_EPSG_4326':[],
         'breedtegraad_EPSG_4326':[],
         }
    )

In [343]:
data = d['mivconfig'][1:] #eerste lijn skippen om enkel de meetpunten te bekomen

Itereren over dictionairy en data in dataframe laden

In [344]:
for i in range(len(data)):
    unieke_id = int(data[i]['@unieke_id'])
    beschrijvende_id = data[i]['meetpunt'][0]['beschrijvende_id']
    volledige_naam = data[i]['meetpunt'][1]['volledige_naam']
    Ident_8 = data[i]['meetpunt'][2]['Ident_8']
    Rijstrook = data[i]['meetpunt'][5]['Rijstrook']
    lengtegraad_EPSG_4326 = float(data[i]['meetpunt'][8]['lengtegraad_EPSG_4326'].replace(',', '.'))
    breedtegraad_EPSG_4326 = float(data[i]['meetpunt'][9]['breedtegraad_EPSG_4326'].replace(',', '.'))
    
    df = pd.DataFrame(
        {'unieke_id':[unieke_id],
         'beschrijvende_id':[beschrijvende_id],
         'volledige_naam':[volledige_naam],
         'Ident_8':[Ident_8],
         'Rijstrook': [Rijstrook],
         'lengtegraad_EPSG_4326':[lengtegraad_EPSG_4326],
         'breedtegraad_EPSG_4326':[breedtegraad_EPSG_4326],
        }
    )

    df_meetpunten = pd.concat([df_meetpunten, df], ignore_index=True)

Dataset cleanen 

In [345]:
df_meetpunten['unieke_id'] = df_meetpunten['unieke_id'].astype(int) #id in juist fromaat zetten
df_meetpunten = df_meetpunten[(df_meetpunten['Ident_8'] == 'R0010001') | (df_meetpunten['Ident_8'] =='R0010002')] #filteren met mask op meetpunten van ring 1 en ring 2
df_meetpunten = df_meetpunten.set_index(pd.Index([x for x in range(len(df_meetpunten))])) #index terug aflopend maken, deze stap was meer voor later te testen
#df_meetpunten.head()

Visualisatie van de meetpunten via folium map

In [346]:
# m = folium.Map(location=[51.2223, 4.3960], zoom_start=10)

# for i in range(len(df_meetpunten)):
#     lat = np.float64(df_meetpunten["breedtegraad_EPSG_4326"][i])
#     long = np.float64(df_meetpunten["lengtegraad_EPSG_4326"][i])
#     u_id = df_meetpunten["Ident_8"][i]
#     #u_id = df_meetpunten.index[i]

#     folium.Marker(
#         location=[lat, long],
#         popup=u_id
#     ).add_to(m)

# m

###  1.2 Edges 

In [347]:
api = "https://geodata.antwerpen.be/arcgissql/rest/services/P_Portal/portal_publiek4/MapServer/297/query?where=WEGNUMMER%20%3D%20'R1'%20AND%20WEGCAT%20%3D%20'hoofdweg'&outFields=WS_OIDN,B_WK_OIDN,E_WK_OIDN,STATUS,MORF,WEGCAT,LSTRNMID,LSTRNM,RSTRNMID,RSTRNM,BEHEER,METHODE,OPNDATUM,BEGINTIJD,BEGINORG,TGBEP,WS_UIDN,WS_GIDN,GBKA_ID,WEGNUMMER,WEGKLASSE,SNELHEID,RIJRICHTING_AUTO,CAT_MOBILITEITSPLAN,BOVENLOKAAL,LABEL,Shape_Length&outSR=4326&f=json"

In [348]:
response = requests.get(api, verify=False)
print(response)

/usr/local/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'geodata.antwerpen.be'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


<Response [200]>


In [349]:
response_data = response.json()
#response_data['features'][0] #kijken hoe data er uit ziet

In [350]:
df_edges_attributes = gpd.GeoDataFrame([response_data['features'][x]['attributes'] for x in range(len(response_data['features']))]) #parsen van edge data
df_edges_geometry = gpd.GeoDataFrame([response_data['features'][x]['geometry'] for x in range(len(response_data['features']))]) #parsen van geometry edge
df_edges_combined = gpd.GeoDataFrame(pd.concat([df_edges_attributes, df_edges_geometry], axis=1)) #df concat

In [351]:
mask = ['WS_OIDN','B_WK_OIDN','E_WK_OIDN', 'SNELHEID', 'RIJRICHTING_AUTO', 'Shape_Length', 'paths'] #gewenste kolommen

In [352]:
df_edges = df_edges_combined[mask]
df_edges.head(1) 

,WS_OIDN,B_WK_OIDN,E_WK_OIDN,SNELHEID,RIJRICHTING_AUTO,Shape_Length,paths
0,421566,2085123,1079886,100,enkel (mee),782.191221,"[[[4.421230557666013, 51.19099637960401], [4.4..."


Coordinaten staan in de verkeerde volgorde

In [353]:
def change_coord(coordinates):
    """
    Deze functie zal coordinaten wisselen van plaats [long, lat] -> [lat, long]
    """

    if isinstance(coordinates[0], list):
        return [change_coord(line) for line in coordinates]
    elif isinstance(coordinates[0], float):  
        return [coordinates[1], coordinates[0]]

In [354]:
df_edges.loc[:, 'paths'] = pd.Series([change_coord(linestring) for linestring in df_edges['paths']]) #kolom wisselen met paths in juist formaat

In [355]:
# m = folium.Map(location=[51.2223, 4.3960], zoom_start=10)

# for i, row in df_edges.iterrows():
#     polyline = folium.PolyLine(row.loc['paths'], color='blue', weight=2.5, opacity=1)
#     popup = folium.Popup(str(row.loc['WS_OIDN']), max_width=300)
#     polyline.add_child(popup)
#     polyline.add_to(m)

# m

Er zijn nog een paar edges in de dataset die weg mogen

In [356]:
removed_edges = [556092, 477787, 411870, 555120, 1177710, 1154275, 
                 1154268, 1154267, 1198422, 1198444, 421567, 536729, 
                 539943, 478156, 421594, 421595, 475072, 536365,
                 499035, 545294, 475084, 555690, 1192085, 1192086,
                 1154254, 1154280, 1154255, 1154257, 1154259, 1198423,
                 1198443, 1154258, 1154284, 1154285
                ]

df_edges = df_edges[[edge not in removed_edges for edge in df_edges['WS_OIDN']]] 

#df_edges.head()

## 2. map matching van meetpunten aan een edge

Om de gemiddelde snelheid van een edge te bekomen moet ik de meetpunten mapmatchen met de respectievelijke edges. Hiervoor zal ik een functie gebruiken die over de meetpuntlocaties (lat, long) zal ittereren en de boldriekhoeksafstand zal bereken van de individuele locaties (lat, long) van de (multi)linestrings van de edges. De edge id van de kortste afstand zal ik dan linken met het meetpunt. 

In [357]:
def switch_edge_same_end(edge):
    """
    Functie om de edges die in de verkeerde richting staan om te draaien
    """
    #bepalen nodes van edge
    end_node_edge = df_edges.loc[df_edges['WS_OIDN'] ==  edge, 'E_WK_OIDN'].iloc[0]
    begin_node_edge = df_edges.loc[df_edges['WS_OIDN'] ==  edge, 'B_WK_OIDN'].iloc[0]

    #tellen of er nodes 2 keer gebruikt worden als begin of einde
    end_node_count = df_edges[df_edges['E_WK_OIDN'] == end_node_edge].count().iloc[0]
    begin_node_count = df_edges[df_edges['B_WK_OIDN'] == begin_node_edge].count().iloc[0]

    #indien er geen nodes dubbel gebruikt worden functie eindigen met False
    if end_node_count < 2 and begin_node_count < 2:
        return False

    #stelt edge in met gemeenschappelijke node
    if end_node_count > 1:
        same_node_edge = df_edges.loc[(df_edges['WS_OIDN'] != edge) & (df_edges['E_WK_OIDN'] == end_node_edge), 'WS_OIDN'].iloc[0]
    elif begin_node_count > 1:
        same_node_edge = df_edges.loc[(df_edges['WS_OIDN'] != edge) & (df_edges['B_WK_OIDN'] == begin_node_edge), 'WS_OIDN'].iloc[0]

    #zal van edge van gemeenschappelijke nodes de nodes wisslen en True returnen
    end_node = df_edges.loc[df_edges['WS_OIDN']==same_node_edge, 'B_WK_OIDN'].iloc[0]
    begin_node = df_edges.loc[df_edges['WS_OIDN']==same_node_edge, 'E_WK_OIDN'].iloc[0]

    df_edges.loc[df_edges['WS_OIDN']==same_node_edge, 'B_WK_OIDN'] = begin_node
    df_edges.loc[df_edges['WS_OIDN']==same_node_edge, 'E_WK_OIDN'] = end_node
    
    return True

def follow_available_route(start_edge_id, rijrichting):
    """
    Functie die de route (lijst van opeenvolgende edge id's) zal returnen met bijhorden info zoals de afstand, en max snelheid  
    """
    #aangezien beide rijvakken dezelfde 'richting' hebben in de dataset, moeten er een ondersheid gemaakt worden 
    if rijrichting == 'r':
        begin_node_id = 'B_WK_OIDN'
        end_node_id = 'E_WK_OIDN'
    elif rijrichting == 'l':
        begin_node_id = 'E_WK_OIDN'
        end_node_id = 'B_WK_OIDN'

    route = [np.int64(start_edge_id)]

    #eerste edge variabelen bepalen
    begin_node = df_edges.loc[df_edges['WS_OIDN']==start_edge_id, begin_node_id].iloc[0]
    end_node = df_edges.loc[df_edges['WS_OIDN']==start_edge_id, end_node_id].iloc[0]
    shape_length = df_edges.loc[df_edges['WS_OIDN']==start_edge_id, 'Shape_Length'].iloc[0] / 1000
    max_snelheid = df_edges.loc[df_edges['WS_OIDN']==start_edge_id, 'SNELHEID'].iloc[0]

    min_steaming_time =  shape_length / np.int32(max_snelheid)
    distance = np.float64(shape_length)
    next_node_available = True

    #while loop die volgende edge zal zoeken gebaseerd op een gemeenschappelijke node  
    while next_node_available:

        edge_id = df_edges.loc[df_edges[begin_node_id]==end_node, 'WS_OIDN'].iloc[0]
        shape_length = df_edges.loc[df_edges['WS_OIDN']==edge_id, 'Shape_Length'].iloc[0] / 1000
        max_snelheid = df_edges.loc[df_edges['WS_OIDN']==edge_id, 'WS_OIDN'].iloc[0]

        min_steaming_time += shape_length / np.int32(max_snelheid)
        distance += np.float64(shape_length)

        end_node = df_edges.loc[df_edges['WS_OIDN']==edge_id, end_node_id].iloc[0]
        route.append(edge_id)
        
        if not end_node in df_edges[begin_node_id].values:
            
            switched_other_edge = switch_edge_same_end(edge_id)
            
            if not switched_other_edge:
                next_node_available = False

    return {'min_steaming_time': min_steaming_time,'distance': distance, 'max_snelheid':min_steaming_time, 'route': route}

In [358]:
ring_1 = follow_available_route(458238, 'r') #route van ring 1 
#ring_1 

In [359]:
ring_2 = follow_available_route(1177708, 'l') #route van ring 2
# ring_2

In [360]:
def degree_to_radian(degree):
    """ 
    Zal graden omzetten naar radialen
    """
    
    coversion_factor = math.pi / np.float64(180)
    radian = degree * coversion_factor
    return radian

def haversing_distance(point1, point2):
    """ 
    Zal de kortste weg op de aardbol berekenen
    """
    
    radius = np.float64(6378000) 
    
    lat1 = degree_to_radian(np.float64(point1[0])) 
    long1 = degree_to_radian(np.float64(point1[1]))

    lat2 = degree_to_radian(np.float64(point2[0]))
    long2 = degree_to_radian(np.float64(point2[1]))

    delta_lat_div_two = (lat1 - lat2) / 2
    delta_long_div_two = (long1 - long2) / 2

    cos_lat1 = math.cos(lat1)
    cos_lat2 = math.cos(lat2)

    delta_lat_sin_pow = pow(math.sin(delta_lat_div_two), 2)
    delta_long_sin_pow = pow(math.sin(delta_long_div_two), 2)

    distance = 2 * radius * math.asin(math.sqrt(delta_lat_sin_pow + cos_lat1 * cos_lat2 * delta_long_sin_pow))

    return distance

In [361]:
#df_meetpunten.head()

In [362]:
#df_edges.head()

In [363]:
def shortest_distance_2_edge(edge, point):
    """ 
    Zal de kortste weg tussen edge en point berekenen
    """
    shortest_distance = 0
    i = 0

    #ittereert over elke coordinaat van de vlakke lijst en zal de kortste afstand opslagen
    for coord in edge:
        distance = haversing_distance(point, coord)
        if i == 0:
            shortest_distance = distance
        elif distance < shortest_distance:
            shortest_distance = distance
        i += 1
    return shortest_distance

def map_matcher():
    """ 
    Zal een dict returnen die de meetpunten mapmatched met de edges
    """
    linked_meetpunten_edges = {}

    #itteratie van elk meetpunt
    for i, meetpunt_row in df_meetpunten.iterrows():
        
        meetpunt_id = meetpunt_row.loc['unieke_id'] #id
        ident_8 = meetpunt_row.loc['Ident_8'] #rijrichting categorie
        meetpunt_coordinates = (meetpunt_row.loc['breedtegraad_EPSG_4326'], meetpunt_row.loc['lengtegraad_EPSG_4326']) #coordinaten

        #if statement die categorie rijrichting zal matchen met categorie edges
        if ident_8 == 'R0010002':
            rij_richting = 'enkel (mee)' 
        else:
            rij_richting = 'enkel (tegen)'
        
        for j, edge_row in df_edges.iterrows():
            
            flat_edge = np.concat(edge_row.loc['paths'])
            edge_id = edge_row.loc['WS_OIDN']
            rijrichting_auto = edge_row.loc['RIJRICHTING_AUTO']

            #deze if zal ervoor zorgen dat enkel meetpunten van de correcte rijstrook gemapmatched kunnen worden
            if rij_richting != rijrichting_auto:
                continue

            #korste afstand van deze edge met het meetpunt
            distance = shortest_distance_2_edge(flat_edge, meetpunt_coordinates)

            #zal dict maken met meetpunt_id als key en tuple met edge_id en afstand 
            if meetpunt_id not in linked_meetpunten_edges.keys():
                linked_meetpunten_edges[meetpunt_id] = (edge_id, distance)
            elif linked_meetpunten_edges[meetpunt_id][1] > distance:
                linked_meetpunten_edges[meetpunt_id] = (edge_id, distance)

    return linked_meetpunten_edges

In [364]:
matched_dict = map_matcher() #dictionairy met de link tussen de meetpunten en edges 
# matched_dict

Visuele check mapmatcher

In [365]:
# meetpunt_id = 3744	
# edge_id = matched_dict[meetpunt_id][0]

# m = folium.Map(location=[51.2223, 4.3960], zoom_start=10)

# lat = df_meetpunten[df_meetpunten["unieke_id"]== meetpunt_id].loc[:,"breedtegraad_EPSG_4326"]
# long = df_meetpunten[df_meetpunten["unieke_id"]== meetpunt_id].loc[:,"lengtegraad_EPSG_4326"]

# folium.Marker(
#     location=[lat, long],
# ).add_to(m)

# polyline = folium.PolyLine(df_edges[df_edges["WS_OIDN"]== edge_id].loc[:, "paths"].iloc[0], color='blue', weight=2.5, opacity=1)
# popup = folium.Popup(str(df_edges[df_edges["WS_OIDN"]== edge_id].loc[:,'WS_OIDN']), max_width=300)
# polyline.add_child(popup)
# polyline.add_to(m)

# m 

In [366]:
df_meetpunten['matched_edge_id'] = [matched_dict[df_meetpunten['unieke_id'][x]][0] for x in range(len(df_meetpunten))]
#df_meetpunten

## 3. Opladen van data naar Postgres database 

In [367]:
load_dotenv()

True

In [368]:
conn = psycopg2.connect(
    host=os.getenv('DB_HOST'),
    dbname=os.getenv('POSTGRES_DB'),
    user=os.getenv('POSTGRES_USER'),
    password=os.getenv('POSTGRES_PASSWORD'),
    port=os.getenv('DB_PORT')
)   
cur = conn.cursor()

In [369]:
#tabel aanmaken
cur.execute("""CREATE TABLE IF NOT EXISTS measurement_points( 
    measurement_id INT PRIMARY KEY,
    describing_id VARCHAR(7),
    full_name TEXT,
    id_8 VARCHAR(8),
    lane VARCHAR(5),
    latitude_measurement_point DOUBLE PRECISION,
    longitude_measurement_point DOUBLE PRECISION,
    matching_edge_id INT
)

""")

conn.commit()

In [370]:
cur.execute("""SELECT measurement_id FROM measurement_points""")
meetpunten_sql = cur.fetchall()
meetpunten_sql = [x[0] for x in meetpunten_sql] 

In [371]:
#tabel invullen 
for i, row in df_meetpunten.iterrows():

    #indien id al bestaat next meetpunt
    if row.loc['unieke_id'] in meetpunten_sql: 
        continue
    
    values = (row.loc['unieke_id'], row.loc['beschrijvende_id'], row.loc['volledige_naam'], row.loc['Ident_8'], row.loc['Rijstrook'], row.loc['lengtegraad_EPSG_4326'], row.loc['breedtegraad_EPSG_4326'], row.loc['matched_edge_id'])
    cur.execute("""INSERT INTO measurement_points (measurement_id, describing_id, full_name, id_8, lane, latitude_measurement_point, 
    longitude_measurement_point, matching_edge_id) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""", values)

conn.commit()

In [372]:
cur.execute("""CREATE TABLE IF NOT EXISTS ring(
    ring_id SMALLINT PRIMARY KEY,
    min_steaming_time DOUBLE PRECISION,
    distance DOUBLE PRECISION,
    max_snelheid DOUBLE PRECISION,
    route INT[]
)""")

conn.commit()

In [373]:
cur.execute("DELETE FROM ring")

values_1 = (1, float(ring_1['min_steaming_time']), float(ring_1['distance']), float(ring_1['max_snelheid']), [int(num) for num in ring_1['route']])
values_2 = (2, float(ring_2['min_steaming_time']), float(ring_2['distance']), float(ring_2['max_snelheid']), [int(num) for num in ring_2['route']])

cur.execute("""INSERT INTO ring(ring_id, min_steaming_time, distance, max_snelheid, route) VALUES (%s, %s, %s, %s, %s)""",values_1)
cur.execute("""INSERT INTO ring(ring_id, min_steaming_time, distance, max_snelheid, route) VALUES (%s, %s, %s, %s, %s)""",values_2)

conn.commit()

In [374]:
cur.close
conn.close()